# Training Pipeline for Differentiated Thyroid Cancer Recurrence
This notebook replicates the training process described in the research paper *Machine learning for risk stratification of thyroid cancer patients: a 15‑year cohort study*. 

We evaluate multiple models (SVM, KNN, Decision Tree, Random Forest, ANN) on three feature subsets:
1. **Clinical/Demographic (No ATA Risk)**
2. **ATA Risk Only**
3. **All Features (Combined)**

Metric reported: Sensitivity, Specificity, PPV, NPV, Accuracy, AUC.


In [1]:
# %pip install torch torchvision --index-url https://download.pytorch.org/whl/cu130

In [2]:
import pandas as pd
import numpy as np
import torch
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, roc_auc_score, recall_score, precision_score

pd.set_option('display.max_columns', None)


In [3]:
# Load preprocessed data
train_path = "./data/preprocessed/1_differentiated_thyroid_cancer_recurrence_train_preprocessed.csv"
test_path = "./data/preprocessed/1_differentiated_thyroid_cancer_recurrence_test_preprocessed.csv"

df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)

LABEL = "Recurred"

y_train = df_train[LABEL]
X_train_full = df_train.drop(columns=[LABEL])

y_test = df_test[LABEL]
X_test_full = df_test.drop(columns=[LABEL])

# Identify ATA Risk columns (OneHotEncoded)
ata_cols = [c for c in X_train_full.columns if "Risk" in c]
print("ATA Risk Columns:", ata_cols)

# Define Feature Sets
# 1. No ATA Risk
X_train_no_ata = X_train_full.drop(columns=ata_cols)
X_test_no_ata = X_test_full.drop(columns=ata_cols)

# 2. ATA Risk Only
X_train_ata = X_train_full[ata_cols]
X_test_ata = X_test_full[ata_cols]

# 3. All Features
X_train_all = X_train_full
X_test_all = X_test_full

datasets = {
    "No ATA": (X_train_no_ata, X_test_no_ata),
    "ATA Only": (X_train_ata, X_test_ata),
    "All Features": (X_train_all, X_test_all)
}

print(f"Train shapes: No ATA {X_train_no_ata.shape}, ATA {X_train_ata.shape}, All {X_train_all.shape}")


ATA Risk Columns: ['cat__Risk_High', 'cat__Risk_Intermediate', 'cat__Risk_Low']
Train shapes: No ATA (283, 48), ATA (283, 3), All (283, 51)


# Model training and evaluation (paper-style metrics)
This section evaluates classic models on three feature subsets (No ATA, ATA Only, All Features). The ANN is trained only on All Features, matching the paper.

In [4]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, accuracy_score, roc_auc_score
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

def _score_values(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        return model.decision_function(X)
    return None

def compute_metrics(y_true, y_pred, y_score=None):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) else 0.0
    specificity = tn / (tn + fp) if (tn + fp) else 0.0
    ppv = tp / (tp + fp) if (tp + fp) else 0.0
    npv = tn / (tn + fn) if (tn + fn) else 0.0
    accuracy = accuracy_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_score) if y_score is not None else np.nan
    return {
        "Sensitivity": sensitivity,
        "Specificity": specificity,
        "PPV": ppv,
        "NPV": npv,
        "Accuracy": accuracy,
        "AUC": auc,
    }

models = {
    "SVM": SVC(kernel="rbf", C=1.0, gamma="scale", probability=True, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
}

ann_model = MLPClassifier(
    hidden_layer_sizes=(37,),
    activation="relu",
    solver="adam",
    alpha=1e-4,
    learning_rate_init=0.001,
    max_iter=500,
    random_state=42,
 )

results = []
for dataset_name, (X_tr, X_te) in datasets.items():
    for model_name, model in models.items():
        model.fit(X_tr, y_train)
        y_pred = model.predict(X_te)
        y_score = _score_values(model, X_te)
        metrics = compute_metrics(y_test, y_pred, y_score)
        results.append({
            "Dataset": dataset_name,
            "Model": model_name,
            **metrics,
        })

# ANN only on All Features
X_tr_all, X_te_all = datasets["All Features"]
ann_model.fit(X_tr_all, y_train)
ann_pred = ann_model.predict(X_te_all)
ann_score = _score_values(ann_model, X_te_all)
ann_metrics = compute_metrics(y_test, ann_pred, ann_score)
results.append({
    "Dataset": "All Features",
    "Model": "ANN",
    **ann_metrics,
})

results_df = pd.DataFrame(results)
percent_cols = ["Sensitivity", "Specificity", "PPV", "NPV", "Accuracy", "AUC"]
results_df[percent_cols] = results_df[percent_cols] * 100
results_df = results_df.sort_values(["Dataset", "Model"]).reset_index(drop=True)

display(results_df.style.format({c: "{:.2f}" for c in percent_cols}))

d:\__Projects\quantum-benchmarking\.venv312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


,Dataset,Model,Sensitivity,Specificity,PPV,NPV,Accuracy,AUC
0,ATA Only,Decision Tree,89.29,87.50,73.53,95.45,88.00,90.18
1,ATA Only,KNN,89.29,87.50,73.53,95.45,88.00,90.18
2,ATA Only,Random Forest,89.29,87.50,73.53,95.45,88.00,90.18
3,ATA Only,SVM,89.29,87.50,73.53,95.45,88.00,90.18
4,All Features,ANN,75.00,93.06,80.77,90.54,88.00,92.49
5,All Features,Decision Tree,75.00,93.06,80.77,90.54,88.00,84.03
6,All Features,KNN,75.00,95.83,87.50,90.79,90.00,94.32
7,All Features,Random Forest,82.14,95.83,88.46,93.24,92.00,95.44
8,All Features,SVM,82.14,94.44,85.19,93.15,91.00,92.19
9,No ATA,Decision Tree,78.57,83.33,64.71,90.91,82.00,80.95
